# EkaQuant Sensitivity Profiling
This notebook calculates layer-wise sensitivity for an LLM across all 11 Indic languages in MMLU-IN using Fisher Information and Perturbation metrics.

In [ ]:
!pip install -q transformers bitsandbytes accelerate peft datasets numpy scipy tqdm matplotlib seaborn

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
except Exception:
    print("Warning: HF_TOKEN not found.")

!git clone https://github.com/TarunNagarajan/EkaQuant.git
%cd EkaQuant
!pip install -e .
%cd ..

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from ekaquant.sensitivity import (
    compute_fisher,
    compute_perturbation_sensitivity,
    compute_magnitude,
)
import matplotlib.pyplot as plt
import json

model_id = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading model {model_id} in 8-bit mode for sensitivity analysis...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, load_in_8bit=True, device_map="auto"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
indic_langs = ["hi", "bn", "kn", "en", "gu", "ml", "mr", "or", "pa", "ta", "te"]
calibration_texts = []
samples_per_lang = 20

print("Fetching calibration samples across all languages...")
for lang in indic_langs:
    try:
        ds = load_dataset(
            "sarvamai/mmlu-indic", name=lang, split="test", trust_remote_code=True
        )
        # Use the question as the context to evaluate
        samples = ds.select(range(min(samples_per_lang, len(ds))))
        for sample in samples:
            calibration_texts.append(sample["question"])
    except Exception as e:
        print(f"Could not load {lang}: {e}")

print(f"Total calibration samples: {len(calibration_texts)}")

In [ ]:
print("Computing Fisher Sensitivity...")
fisher_sensitivity = compute_fisher(
    model, tokenizer, calibration_texts, clip_samples=16
)
with open("fisher_sensitivity.json", "w") as f:
    json.dump(fisher_sensitivity, f)

print("Computing Perturbation Sensitivity...")
perturbation_sensitivity = compute_perturbation_sensitivity(
    model, tokenizer, calibration_texts
)
with open("perturbation_sensitivity.json", "w") as f:
    json.dump(perturbation_sensitivity, f)

print("Computing Magnitude Sensitivity...")
magnitude_sensitivity = compute_magnitude(model)
with open("magnitude_sensitivity.json", "w") as f:
    json.dump(magnitude_sensitivity, f)

In [ ]:
import pandas as pd
import seaborn as sns


def extract_layer_idx(name):
    parts = name.split(".")
    for p in parts:
        if p.isdigit():
            return int(p)
    return -1


data = []
for method, sens_dict in [
    ("Fisher", fisher_sensitivity),
    ("Perturbation", perturbation_sensitivity),
    ("Magnitude", magnitude_sensitivity),
]:
    for layer, score in sens_dict.items():
        if "layers" in layer and ("mlp" in layer or "self_attn" in layer):
            data.append(
                {
                    "Layer": layer,
                    "Score": score,
                    "Method": method,
                    "LayerIdx": extract_layer_idx(layer),
                }
            )

df = pd.DataFrame(data)

# Normalize scores within each method for comparison
for method in df["Method"].unique():
    mask = df["Method"] == method
    df.loc[mask, "NormalizedScore"] = (
        df.loc[mask, "Score"] / df.loc[mask, "Score"].max()
    )

plt.figure(figsize=(15, 6))
sns.lineplot(data=df, x="LayerIdx", y="NormalizedScore", hue="Method")
plt.title("Layer Sensitivity Comparison (Normalized)")
plt.xlabel("Transformer Layer Index")
plt.ylabel("Normalized Sensitivity Score")
plt.savefig("sensitivity_comparison.png")
plt.show()